# 04 — Feature Analysis (3-Stage Band Selection)

A novel 3-stage pipeline to select the optimal spectral–temporal band subset for semantic segmentation.

| Stage | Method | Cost | Output |
|---|---|---|---|
| 1 — Filter | GSI + RF importance → joint score | Very low | Top ~25 candidate channels |
| 2 — CNN Forward Selection | Lightweight U-Net as wrapper oracle | Medium | ~10–15 selected channels |
| 3 — Full Validation | U-Net / DeepLabV3+ / SegFormer | High | mIoU comparison table |

**Assumes `02_image_processing` has been run.** Preprocessed chips (images + masks) must exist in `data/processed/`.

## Configuration

In [ ]:
import sys
sys.path.append("../")

# ── Raster paths (pixel-level sampling for Stage 1) ──────────────────────────
S2_PATH    = "../data/raw/images/S2H_2023_2023_06_30_nodata_clipped.tif"
LABEL_PATH = "../data/raw/cdl/2023_30m_cdls_10m_clipped.tif"

# ── Chip dataset paths (for Stage 2 CNN training) ────────────────────────────
CHIPS_DIR  = "../data/processed/crop_mapping_multi_images_subset"
IMAGES_DIR = f"{CHIPS_DIR}/images"
MASKS_DIR  = f"{CHIPS_DIR}/masks"

# ── Stage 1 parameters ───────────────────────────────────────────────────────
SAMPLE_FRACTION = 0.005   # 0.5% pixel sample for GSI/RF
GSI_RF_ALPHA    = 0.5     # weight for GSI in joint score (1-alpha = RF weight)
TOP_K           = 25      # candidate channels passed to Stage 2

# ── Stage 2 parameters ───────────────────────────────────────────────────────
S2_ENCODER      = "resnet18"    # lightweight encoder
S2_PATCH_SIZE   = 128
S2_EPOCHS       = 15            # fast approximate training
S2_PATIENCE     = 5             # early stopping
S2_DELTA        = 0.005         # min mIoU gain to accept a new band
S2_NO_IMPROVE   = 3             # stop if no improvement for N consecutive bands
S2_MAX_BANDS    = 15            # hard cap on selected bands

# ── Stage 3 parameters ───────────────────────────────────────────────────────
S3_EPOCHS       = 100
S3_PATCH_SIZE   = 256
NUM_CLASSES     = 8             # including background

---
# Stage 1 — Filter Preselection
**Goal:** Reduce all channels to ~25 candidates using cheap, non-spatial statistics.

$$\text{Score}_b = \alpha \cdot \text{GSI}_{\text{norm},b} + (1-\alpha) \cdot \text{RF}_{\text{norm},b}$$

## 1.1 — Load & Sample Pixels

In [ ]:
import importlib
import utils.band_selection as bs
importlib.reload(bs)

df, band_names = bs.load_raster_data(S2_PATH, LABEL_PATH, SAMPLE_FRACTION)
CLASS_COL = "class_label"

print(f"Sampled pixels : {len(df):,}")
print(f"Bands          : {band_names}")
print(f"Classes        : {sorted(df[CLASS_COL].unique())}")

## 1.2 — Compute GSI

In [ ]:
print("Computing GSI...")
gsi_df   = bs.calculate_gsi(df, CLASS_COL)
gsi_mean = gsi_df.mean(axis=1).sort_values(ascending=False)

print("\nGSI ranking (top 10):")
print(gsi_mean.head(10).to_string())
bs.plot_gsi(gsi_df, save_path="gsi_plot.png")

## 1.3 — Compute RF Feature Importance

In [ ]:
import matplotlib.pyplot as plt

print("Computing RF feature importance...")
rf_importance = bs.compute_rf_importance(df, CLASS_COL, n_estimators=200)

print("\nRF importance ranking (top 10):")
print(rf_importance.head(10).to_string())

rf_importance.sort_values().plot(kind="barh", figsize=(8, 5), color="steelblue")
plt.title("RF Feature Importance per Band")
plt.xlabel("Normalized Importance")
plt.tight_layout()
plt.show()

## 1.4 — Compute Joint Score & Select Top K

In [ ]:
import pandas as pd

joint_score = bs.compute_joint_score(gsi_mean, rf_importance, alpha=GSI_RF_ALPHA)
stage1_bands = bs.select_top_k(joint_score, k=TOP_K)

# Summary table
summary = pd.DataFrame({
    "GSI (norm)":   (gsi_mean - gsi_mean.min()) / (gsi_mean.max() - gsi_mean.min()),
    "RF (norm)":    rf_importance,
    "Joint Score":  joint_score,
}).sort_values("Joint Score", ascending=False)
summary["Selected"] = summary.index.isin(stage1_bands)

print(f"Stage 1 selected {len(stage1_bands)} bands: {stage1_bands}")
print()
print(summary.to_string())

# Plot joint score
summary["Joint Score"].sort_values().plot(kind="barh", figsize=(8, 5), color="darkorange")
plt.axvline(joint_score[stage1_bands[-1]], color="red", linestyle="--", label=f"Top-{TOP_K} cutoff")
plt.title(f"Joint Score per Band (α={GSI_RF_ALPHA})")
plt.xlabel("Joint Score")
plt.legend()
plt.tight_layout()
plt.show()

---
# Stage 2 — CNN Forward Selection
**Goal:** Use a lightweight U-Net as the selection oracle. Iteratively add bands, keeping only those that raise validation mIoU by at least δ.

$$\text{keep band } b \iff \text{mIoU}_{\text{new}} > \text{mIoU}_{\text{prev}} + \delta$$

> ⚠️ Each iteration trains a small model for ~15 epochs. Expect ~6–10 hours total.

## 2.1 — Setup

In [ ]:
import os, time, json
import numpy as np
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader, random_split
import rasterio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Map band name → 0-indexed position in the stacked image chips
band_index = {name: i for i, name in enumerate(band_names)}

In [ ]:
from torch.utils.data import Dataset

class ChipDataset(Dataset):
    """Load image chips and masks; optionally restrict to a band subset."""
    def __init__(self, images_dir, masks_dir, band_indices=None):
        self.img_paths  = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith(".tif")])
        self.mask_paths = sorted([os.path.join(masks_dir,  f) for f in os.listdir(masks_dir)  if f.endswith(".tif")])
        self.band_indices = band_indices  # None = all bands

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32)  # (C, H, W)
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.int64)  # (H, W)

        if self.band_indices is not None:
            img = img[self.band_indices]

        # Min-max normalize per chip
        for c in range(img.shape[0]):
            mn, mx = img[c].min(), img[c].max()
            img[c] = (img[c] - mn) / (mx - mn + 1e-9)

        return torch.from_numpy(img), torch.from_numpy(mask)

In [ ]:
def build_unet(in_channels, num_classes, encoder=S2_ENCODER):
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights=None,      # no pretrained weights — band count varies
        in_channels=in_channels,
        classes=num_classes,
    ).to(DEVICE)


def compute_miou(preds, labels, num_classes, ignore_index=0):
    ious = []
    preds_flat  = preds.view(-1).cpu().numpy()
    labels_flat = labels.view(-1).cpu().numpy()
    for cls in range(1, num_classes):  # skip background
        pred_mask  = preds_flat  == cls
        label_mask = labels_flat == cls
        intersection = (pred_mask & label_mask).sum()
        union        = (pred_mask | label_mask).sum()
        if union > 0:
            ious.append(intersection / union)
    return float(np.mean(ious)) if ious else 0.0


def train_eval(band_indices, num_classes, epochs=S2_EPOCHS, patience=S2_PATIENCE):
    """Train a lightweight U-Net on the given band subset and return val mIoU."""
    dataset  = ChipDataset(IMAGES_DIR, MASKS_DIR, band_indices=band_indices)
    n_val    = max(1, int(0.2 * len(dataset)))
    n_train  = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))

    train_dl = DataLoader(train_ds, batch_size=8,  shuffle=True,  num_workers=2)
    val_dl   = DataLoader(val_ds,   batch_size=8,  shuffle=False, num_workers=2)

    model     = build_unet(len(band_indices), num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

    best_miou, no_improve = 0.0, 0

    for epoch in range(epochs):
        # ── Training ──
        model.train()
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()

        # ── Validation ──
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, masks in val_dl:
                preds = model(imgs.to(DEVICE)).argmax(dim=1)
                all_preds.append(preds.cpu())
                all_labels.append(masks)

        miou = compute_miou(torch.cat(all_preds), torch.cat(all_labels), num_classes)

        if miou > best_miou + 1e-4:
            best_miou, no_improve = miou, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    return best_miou

## 2.2 — Forward Selection Loop

In [ ]:
# Rank Stage 1 candidates by joint score (already sorted)
candidates = stage1_bands  # ordered by joint score descending

selected   = []
prev_miou  = 0.0
no_improve = 0
history    = []   # [(n_bands, miou, band_added, accepted)]

print(f"Forward selection over {len(candidates)} candidates (δ={S2_DELTA}, max_bands={S2_MAX_BANDS})\n")

for band in candidates:
    if len(selected) >= S2_MAX_BANDS:
        print(f"Reached max_bands={S2_MAX_BANDS}. Stopping.")
        break
    if no_improve >= S2_NO_IMPROVE:
        print(f"No improvement for {S2_NO_IMPROVE} consecutive bands. Stopping.")
        break

    trial_bands   = selected + [band]
    trial_indices = [band_index[b] for b in trial_bands]

    t0 = time.time()
    miou = train_eval(trial_indices, NUM_CLASSES)
    elapsed = time.time() - t0

    gain     = miou - prev_miou
    accepted = gain >= S2_DELTA

    if accepted:
        selected  = trial_bands
        prev_miou = miou
        no_improve = 0
        tag = "✅ accepted"
    else:
        no_improve += 1
        tag = "❌ rejected"

    history.append({"n_bands": len(trial_bands), "band": band,
                    "mIoU": miou, "gain": gain, "accepted": accepted})
    print(f"  {tag}  +{band:<6}  mIoU={miou:.4f}  gain={gain:+.4f}  ({elapsed:.0f}s)")

print(f"\n✅ Stage 2 selected {len(selected)} bands: {selected}")

## 2.3 — Plot mIoU vs Band Count

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history)

accepted_df = hist_df[hist_df["accepted"]]
rejected_df = hist_df[~hist_df["accepted"]]

plt.figure(figsize=(10, 5))
plt.plot(accepted_df["n_bands"], accepted_df["mIoU"], "o-", color="green",  label="Accepted band")
plt.scatter(rejected_df["n_bands"], rejected_df["mIoU"], marker="x", color="red", s=80, label="Rejected band")
plt.axhline(prev_miou, linestyle="--", color="gray", label=f"Final mIoU={prev_miou:.4f}")
plt.xlabel("Number of bands")
plt.ylabel("Validation mIoU")
plt.title("CNN Forward Selection: mIoU vs Band Count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("forward_selection_curve.png", dpi=150)
plt.show()

print(hist_df.to_string(index=False))

# Save selection history
stage2_bands = selected
hist_df.to_csv("forward_selection_history.csv", index=False)
print("\nSaved: forward_selection_history.csv")

---
# Stage 3 — Full Model Validation
**Goal:** Train full segmentation models on three band sets and compare performance.

| Experiment | Band Set | Description |
|---|---|---|
| A — Baseline | All bands | No selection |
| B — Stage 1 | Top-K filter bands | GSI + RF only |
| C — Stage 2 | CNN-selected bands | Forward selection |

Each trained with: U-Net · DeepLabV3+ · SegFormer

## 3.1 — Define Experiments

In [ ]:
all_band_indices    = list(range(len(band_names)))           # Exp A
stage1_band_indices = [band_index[b] for b in stage1_bands]  # Exp B
stage2_band_indices = [band_index[b] for b in stage2_bands]  # Exp C

experiments = {
    "A_all":     {"bands": all_band_indices,    "label": f"All ({len(all_band_indices)} bands)"},
    "B_stage1":  {"bands": stage1_band_indices, "label": f"Stage1 Top-{TOP_K} ({len(stage1_band_indices)} bands)"},
    "C_stage2":  {"bands": stage2_band_indices, "label": f"Stage2 CNN-selected ({len(stage2_band_indices)} bands)"},
}

models = ["unet", "deeplabv3plus", "segformer"]

print("Experiments:")
for key, exp in experiments.items():
    print(f"  {key}: {exp['label']}")

## 3.2 — Train & Evaluate

> ⚠️ This will take ~27 GPU hours (3 models × 3 experiments × ~3 hours each). Run on a GPU server.

In [ ]:
import geoai

results = []

for exp_key, exp in experiments.items():
    for model_name in models:
        band_indices = exp["bands"]
        n_channels   = len(band_indices)
        print(f"\n─── {exp_key} | {model_name} | {n_channels} channels ───")

        out_dir = f"../models/{exp_key}_{model_name}"
        os.makedirs(out_dir, exist_ok=True)

        t_start  = time.time()
        mem_before = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

        geoai.train_segmentation_model(
            images_dir=IMAGES_DIR,
            labels_dir=MASKS_DIR,
            output_dir=out_dir,
            architecture=model_name,
            num_channels=n_channels,
            num_classes=NUM_CLASSES,
            batch_size=8,
            num_epochs=S3_EPOCHS,
            learning_rate=1e-4,
            val_split=0.2,
        )

        elapsed   = time.time() - t_start
        mem_after = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

        # Load best val mIoU from training log
        log_path = os.path.join(out_dir, "training_log.json")
        if os.path.exists(log_path):
            with open(log_path) as f:
                log = json.load(f)
            best_miou = max(entry.get("val_miou", 0) for entry in log)
        else:
            best_miou = None

        results.append({
            "experiment":   exp_key,
            "band_label":   exp["label"],
            "model":        model_name,
            "n_channels":   n_channels,
            "val_mIoU":     best_miou,
            "train_time_s": round(elapsed),
            "gpu_mem_mb":   round((mem_after - mem_before) / 1e6, 1),
        })

        print(f"  mIoU={best_miou:.4f}  time={elapsed/3600:.1f}h  mem={results[-1]['gpu_mem_mb']}MB")

## 3.3 — Compare Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("stage3_results.csv", index=False)

# Pivot table: models as columns, experiments as rows
pivot = results_df.pivot_table(
    index=["experiment", "band_label"],
    columns="model",
    values="val_mIoU"
).round(4)

print("\nmIoU Comparison Table")
print(pivot.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mIoU comparison
for model_name, grp in results_df.groupby("model"):
    axes[0].plot(grp["experiment"], grp["val_mIoU"], marker="o", label=model_name)
axes[0].set_title("Validation mIoU by Experiment & Model")
axes[0].set_ylabel("mIoU")
axes[0].set_xlabel("Experiment")
axes[0].legend()
axes[0].grid(True)

# Training time comparison
for model_name, grp in results_df.groupby("model"):
    axes[1].plot(grp["experiment"], grp["train_time_s"] / 3600, marker="s", label=model_name)
axes[1].set_title("Training Time (hours) by Experiment & Model")
axes[1].set_ylabel("Hours")
axes[1].set_xlabel("Experiment")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("stage3_comparison.png", dpi=150)
plt.show()